# 01 – Build train/val/test splits for AlphaEarth chips

This notebook is a thin wrapper around `tools/build_chips_index.py`.

It:
- runs the split builder (or reads its output),
- inspects class balance per split,
- shows a few example rows.

**Expected input:** TIFF chips under `data/derived/alpha_earth_clips/ee/...` (tomato + non_tomato) as generated by the Earth Engine notebooks.

**Outputs:** `data/splits/chips_index.parquet` and `chips_index.csv`.


In [ ]:
import subprocess
from pathlib import Path

import pandas as pd

from src.utils.paths import REPO_ROOT, load_paths_config

cfg = load_paths_config()
splits_dir = REPO_ROOT / cfg.get("data", {}).get("splits", "data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)
index_parquet = splits_dir / "chips_index.parquet"
index_csv = splits_dir / "chips_index.csv"

print("Repo root:", REPO_ROOT)
print("Splits dir:", splits_dir)
print("Index parquet:", index_parquet)
print("Index csv:", index_csv)


## 1. Build (or rebuild) the chips index

Run this cell once to (re)generate the index from local TIFFs. It calls
`python tools/build_chips_index.py` under the hood.


In [ ]:
cmd = ["python", "tools/build_chips_index.py"]
print("Running:", " ".join(cmd))
completed = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print("Return code:", completed.returncode)
print("--- stdout ---")
print(completed.stdout)
print("--- stderr ---")
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError("build_chips_index.py failed; see output above")


## 2. Load the index and inspect class balance


In [ ]:
if index_parquet.is_file():
    df = pd.read_parquet(index_parquet)
elif index_csv.is_file():
    df = pd.read_csv(index_csv)
else:
    raise FileNotFoundError(f"No chips_index.* found in {splits_dir}")

print("Rows:", len(df))
df.head()


In [ ]:
summary = (
    df.groupby(["split", "class_label"])["chip_id"]
    .count()
    .rename("count")
    .reset_index()
    .sort_values(["split", "class_label"])
)
summary


## 3. Sanity check a few entries

This just prints a small sample of paths to confirm they match your expectations.


In [ ]:
df.sample(5, random_state=42)[["chip_id", "class_label", "split", "local_path"]]
